# Defect Segmentation — Output Visualizer

General-purpose notebook for inspecting inference outputs from `3_inference.py`.

**Set the two paths in Cell 2, then run all cells.**

Expected output folder layout (produced by `3_inference.py --otsu-refine`):
```
<OUTPUT_BASE>/                          ← union (cylinder ∪ ring), all tissue
<OUTPUT_BASE>_cylinder/                 ← 10 mm cylinder, all tissue
<OUTPUT_BASE>_ring/                     ← 18 mm ring, all tissue
<OUTPUT_BASE>_cylinder_bone/            ← 10 mm cylinder, bone only (Otsu)
<OUTPUT_BASE>_ring_bone/                ← 18 mm ring, bone only (Otsu)
```
If `--otsu-refine` was not used the last two folders will be absent and the
bone-series cells will be skipped automatically.

## Cell 1 — Imports

In [ ]:
from pathlib import Path
import numpy as np
import pydicom
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import warnings
warnings.filterwarnings('ignore')

## Cell 2 — ⚙️ Configuration (edit these two paths)

In [ ]:
# Path to the original (input) DICOM directory
INPUT_DIR   = Path('/path/to/original/dicom_folder')

# Path to the union output directory produced by 3_inference.py
# (the --output argument you passed to 3_inference.py)
OUTPUT_BASE = Path('/path/to/output_dicom')

# Derived paths (auto, do not edit)
CYL_DIR       = Path(str(OUTPUT_BASE) + '_cylinder')
RING_DIR      = Path(str(OUTPUT_BASE) + '_ring')
CYL_BONE_DIR  = Path(str(OUTPUT_BASE) + '_cylinder_bone')
RING_BONE_DIR = Path(str(OUTPUT_BASE) + '_ring_bone')

HAS_BONE_SERIES = CYL_BONE_DIR.exists() and RING_BONE_DIR.exists()

print(f'Input          : {INPUT_DIR}  (exists: {INPUT_DIR.exists()})')
print(f'Union output   : {OUTPUT_BASE.name}  (exists: {OUTPUT_BASE.exists()})')
print(f'Cylinder       : {CYL_DIR.name}  (exists: {CYL_DIR.exists()})')
print(f'Ring           : {RING_DIR.name}  (exists: {RING_DIR.exists()})')
print(f'Cylinder bone  : {CYL_BONE_DIR.name}  (exists: {CYL_BONE_DIR.exists()})')
print(f'Ring bone      : {RING_BONE_DIR.name}  (exists: {RING_BONE_DIR.exists()})')
print(f'\nOtsu-refined bone series available: {HAS_BONE_SERIES}')

## Cell 3 — Load DICOM Series

In [ ]:
def load_series(directory):
    """Load all .dcm files from a directory, sorted by InstanceNumber."""
    files = sorted(f for f in Path(directory).glob('*.dcm') if not f.name.startswith('._'))
    if not files:
        raise FileNotFoundError(f'No .dcm files found in {directory}')
    slices = []
    for f in files:
        ds = pydicom.dcmread(str(f))
        slices.append((int(ds.InstanceNumber), ds.pixel_array))
    slices.sort(key=lambda x: x[0])
    instances = [s[0] for s in slices]
    arrays    = [s[1] for s in slices]
    return instances, arrays

print('Loading input series...')
instances, input_arrays = load_series(INPUT_DIR)
N = len(instances)
print(f'  {N} slices  |  shape {input_arrays[0].shape}')

print('Loading union output...')
_, union_arrays = load_series(OUTPUT_BASE)

print('Loading cylinder series...')
_, cyl_arrays = load_series(CYL_DIR)

print('Loading ring series...')
_, ring_arrays = load_series(RING_DIR)

if HAS_BONE_SERIES:
    print('Loading cylinder-bone series...')
    _, cyl_bone_arrays = load_series(CYL_BONE_DIR)
    print('Loading ring-bone series...')
    _, ring_bone_arrays = load_series(RING_BONE_DIR)
else:
    cyl_bone_arrays = ring_bone_arrays = None
    print('Bone series not found — skipping (re-run 3_inference.py with --otsu-refine to generate)')

active_indices = [i for i, a in enumerate(union_arrays) if a.any()]
print(f'\nActive predicted slices : {len(active_indices)} / {N}')
if active_indices:
    print(f'Instance range          : {instances[active_indices[0]]} → {instances[active_indices[-1]]}')

## Cell 4 — Volume Summary

In [ ]:
# Read pixel spacing from the output DICOM (already has correct voxel size)
ds_ref = pydicom.dcmread(str(sorted(f for f in OUTPUT_BASE.glob('*.dcm')
                                    if not f.name.startswith('._'))[0]))
ps = [float(v) for v in ds_ref.PixelSpacing]
st = float(getattr(ds_ref, 'SliceThickness', ps[0]))
VOXEL_MM3 = ps[0] * ps[1] * st

def count_nz(arrays):
    return sum((a != 0).sum() for a in arrays)

nz_union = count_nz(union_arrays)
nz_cyl   = count_nz(cyl_arrays)
nz_ring  = count_nz(ring_arrays)

print(f'Voxel size : {ps[0]:.4f} x {ps[1]:.4f} x {st:.4f} mm  =  {VOXEL_MM3:.6f} mm³')
print()
print(f'=== ROI Volume ===')
print(f'Union  (cyl ∪ ring) : {nz_union*VOXEL_MM3:>10.2f} mm³')
print(f'Cylinder (10 mm)    : {nz_cyl*VOXEL_MM3:>10.2f} mm³')
print(f'Ring     (18 mm)    : {nz_ring*VOXEL_MM3:>10.2f} mm³')

if HAS_BONE_SERIES:
    nz_cb = count_nz(cyl_bone_arrays)
    nz_rb = count_nz(ring_bone_arrays)
    print()
    print(f'=== Bone Volume (Otsu-refined) ===')
    print(f'Cylinder bone  : {nz_cb*VOXEL_MM3:>10.2f} mm³  |  BV/TV = {100*nz_cb/max(nz_cyl,1):.1f} %')
    print(f'Ring bone      : {nz_rb*VOXEL_MM3:>10.2f} mm³  |  BV/TV = {100*nz_rb/max(nz_ring,1):.1f} %')

## Cell 5 — Interactive Slice Viewer (Input vs Prediction)

In [ ]:
def vrange(arr):
    nz = arr[arr != 0]
    if nz.size == 0:
        return 0.0, 1.0
    return float(np.percentile(nz, 1)), float(np.percentile(nz, 99))

def show_slice(idx):
    inst  = instances[idx]
    inp   = input_arrays[idx].astype(np.float32)
    union = union_arrays[idx].astype(np.float32)
    cyl   = cyl_arrays[idx].astype(np.float32)
    ring  = ring_arrays[idx].astype(np.float32)

    inp_vmin, inp_vmax = np.percentile(inp[inp > 0], [1, 99]) if (inp > 0).any() else (0, 1)
    is_active = union.any()

    ncols = 5 if HAS_BONE_SERIES else 4
    fig, axes = plt.subplots(1, ncols, figsize=(ncols * 4, 5))
    fig.suptitle(f'Instance {inst}{"  ★ ACTIVE" if is_active else ""}', fontsize=12, fontweight='bold')

    axes[0].imshow(inp, cmap='gray', vmin=inp_vmin, vmax=inp_vmax)
    axes[0].set_title('Input CT')
    axes[0].axis('off')

    vmin, vmax = vrange(union)
    axes[1].imshow(union, cmap='gray', vmin=vmin, vmax=vmax)
    axes[1].set_title('Union output')
    axes[1].axis('off')

    vmin, vmax = vrange(cyl)
    axes[2].imshow(cyl, cmap='gray', vmin=vmin, vmax=vmax)
    axes[2].set_title('Cylinder (10 mm)')
    axes[2].axis('off')

    vmin, vmax = vrange(ring)
    axes[3].imshow(ring, cmap='gray', vmin=vmin, vmax=vmax)
    axes[3].set_title('Ring (18 mm)')
    axes[3].axis('off')

    if HAS_BONE_SERIES:
        cb = cyl_bone_arrays[idx].astype(np.float32)
        vmin, vmax = vrange(cb)
        axes[4].imshow(cb, cmap='hot', vmin=vmin, vmax=vmax)
        axes[4].set_title('Cylinder bone (Otsu)')
        axes[4].axis('off')

    plt.tight_layout()
    plt.show()

default_idx = active_indices[len(active_indices)//2] if active_indices else 0
interact(
    show_slice,
    idx=IntSlider(min=0, max=N-1, step=1, value=default_idx,
                  description='Slice index', continuous_update=False,
                  style={'description_width': 'initial'})
)

## Cell 6 — Active Slices Grid Overview

In [ ]:
if not active_indices:
    print('No active slices found.')
else:
    n_show  = min(25, len(active_indices))
    sampled = active_indices[::max(1, len(active_indices)//n_show)][:n_show]
    cols = 5
    rows = int(np.ceil(len(sampled) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    axes = np.array(axes).flatten()

    for k, idx in enumerate(sampled):
        arr  = cyl_arrays[idx].astype(np.float32)
        inst = instances[idx]
        vmin, vmax = vrange(arr)
        axes[k].imshow(arr, cmap='gray', vmin=vmin, vmax=vmax)
        axes[k].set_title(f'Inst {inst}', fontsize=8)
        axes[k].axis('off')

    for ax in axes[len(sampled):]:
        ax.axis('off')

    fig.suptitle(f'Active slices — cylinder ROI ({len(active_indices)} total)', fontsize=12)
    plt.tight_layout()
    out_png = OUTPUT_BASE.parent / (OUTPUT_BASE.name + '_overview.png')
    plt.savefig(str(out_png), dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_png}')